In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public class Position
{
    public int Row { get; }
    public int Col { get; }
    
    public Position(int row, int col)
    {
        Row = row;
        Col = col;
    }
    
    public override bool Equals(object obj)
    {
        if (obj is Position other)
            return Row == other.Row && Col == other.Col;
        return false;
    }
    
    public override int GetHashCode() => HashCode.Combine(Row, Col);
}

public class CheatState
{
    public Position Pos { get; }
    public int Steps { get; }
    public int CheatTimeLeft { get; }
    public bool IsCheating { get; }
    
    public CheatState(Position pos, int steps, int cheatTimeLeft, bool isCheating)
    {
        Pos = pos;
        Steps = steps;
        CheatTimeLeft = cheatTimeLeft;
        IsCheating = isCheating;
    }
}

public class MazeSolver
{
    private readonly char[][] maze;
    private readonly Position start;
    private readonly Position end;
    private readonly int maxCheatTime;

    public MazeSolver(string[] input, int maxCheatTime)
    {
        maze = input.Select(line => line.ToCharArray()).ToArray();
        start = FindPosition('S');
        end = FindPosition('E');
        this.maxCheatTime = maxCheatTime;
    }

    private Position FindPosition(char target)
    {
        for (int row = 0; row < maze.Length; row++)
            for (int col = 0; col < maze[row].Length; col++)
                if (maze[row][col] == target)
                    return new Position(row, col);
        return null;
    }

    public int CountCheatsWithMinSaving(int minSaving)
    {
        var normalPath = FindShortestPath(start, end, false);
        if (normalPath == -1) return 0;

        var cheats = new HashSet<(Position, Position)>();
        var queue = new Queue<CheatState>();
        var visited = new HashSet<(Position, int, bool)>();

        queue.Enqueue(new CheatState(start, 0, maxCheatTime, false));

        while (queue.Count > 0)
        {
            var current = queue.Dequeue();
            var state = (current.Pos, current.CheatTimeLeft, current.IsCheating);
            
            if (visited.Contains(state)) continue;
            visited.Add(state);

            if (current.Pos.Equals(end))
            {
                int saving = normalPath - current.Steps;
                if (saving >= minSaving)
                    cheats.Add((start, end));
                continue;
            }

            var neighbors = GetNeighbors(current);
            foreach (var next in neighbors)
                queue.Enqueue(next);
        }

        return cheats.Count;
    }

    private int FindShortestPath(Position start, Position end, bool allowCheating)
    {
        var queue = new Queue<(Position, int)>();
        var visited = new HashSet<Position>();
        
        queue.Enqueue((start, 0));
        
        while (queue.Count > 0)
        {
            var (pos, steps) = queue.Dequeue();
            
            if (pos.Equals(end)) return steps;
            if (visited.Contains(pos)) continue;
            
            visited.Add(pos);
            foreach (var next in GetValidMoves(pos, allowCheating))
                queue.Enqueue((next, steps + 1));
        }
        
        return -1;
    }

    private IEnumerable<Position> GetValidMoves(Position pos, bool allowCheating)
    {
        int[] dr = { -1, 1, 0, 0 };
        int[] dc = { 0, 0, -1, 1 };
        
        for (int i = 0; i < 4; i++)
        {
            int newRow = pos.Row + dr[i];
            int newCol = pos.Col + dc[i];
            
            if (newRow >= 0 && newRow < maze.Length && 
                newCol >= 0 && newCol < maze[0].Length &&
                (allowCheating || maze[newRow][newCol] != '#'))
            {
                yield return new Position(newRow, newCol);
            }
        }
    }

    private IEnumerable<CheatState> GetNeighbors(CheatState current)
    {
        int[] dr = { -1, 1, 0, 0 };
        int[] dc = { 0, 0, -1, 1 };
        
        for (int i = 0; i < 4; i++)
        {
            int newRow = current.Pos.Row + dr[i];
            int newCol = current.Pos.Col + dc[i];
            
            if (newRow < 0 || newRow >= maze.Length || 
                newCol < 0 || newCol >= maze[0].Length)
                continue;

            bool isWall = maze[newRow][newCol] == '#';
            var newPos = new Position(newRow, newCol);

            if (!isWall)
            {
                yield return new CheatState(newPos, current.Steps + 1, 
                    current.CheatTimeLeft, current.IsCheating);
            }
            else if (!current.IsCheating && current.CheatTimeLeft > 0)
            {
                yield return new CheatState(newPos, current.Steps + 1, 
                    current.CheatTimeLeft - 1, true);
            }
        }
    }
}